### **Step 1:** Connecting and looking at the flattened data.




In [19]:
import pandas
import duckdb

connection = duckdb.connect('novellia.duckdb')
connection.execute('''
SELECT
    *
FROM
    stg_immunization
''').df()

,immunization_id,status,vaccine_name,vaccine_code,vaccine_code_system,patient_id,encounter_id,occurrence_date_time,primary_source,location
0,0000e3ef-3cf9-572b-f476-6398236b3624,completed,"rotavirus, monovalent",119,http://hl7.org/fhir/sid/cvx,8fb4ba44-2680-3ba1-bd88-d1b3dc36746e,952db305-85b8-55bd-2a71-c46270adb226,2020-05-16T05:15:06-04:00,True,"ROCK RIDGE FAMILY MEDICINE, P.A."
1,000a7057-ecc3-0b40-e99f-17aa1a3f344a,completed,"Influenza, seasonal, injectable, preservative ...",140,http://hl7.org/fhir/sid/cvx,57980abf-174f-e0ae-86f9-60000b4ea5e5,4d108fc2-945b-d475-0f1b-4aa1cb49d8a9,2017-02-12T00:02:43-05:00,True,SWOPE HEALTH SERVICES
2,0010398a-13e1-366a-b7f1-d05ff44b64d2,completed,Hib (PRP-OMP),49,http://hl7.org/fhir/sid/cvx,b18064b3-80f0-dbd6-3f27-969d50b31936,0cb9f3eb-b80a-abdd-f849-67dc7b2bf347,2019-05-25T16:45:10-04:00,True,RESTORATION HEALTH CARE LLC
3,00159ce1-b151-d88d-1151-6b741e011255,completed,"Influenza, seasonal, injectable, preservative ...",140,http://hl7.org/fhir/sid/cvx,aee9e860-9323-3451-2e55-9afc1f09ed82,909f30a8-d009-9c20-4797-68b4e7488076,2021-12-26T14:52:05-05:00,True,ELLSWORTH COUNTY MEDICAL CENTER
4,00194103-776e-f6d8-104e-36e9c7ad6876,completed,"Hep A, adult",52,http://hl7.org/fhir/sid/cvx,cb919cb1-f9de-ba69-bb5e-a5bb098ad5c7,9eb1a58c-6a8e-4475-5ace-cc9b9fd95edf,2021-03-25T18:54:26-04:00,True,TRUSTED PRIMARY CARE
...,...,...,...,...,...,...,...,...,...,...
17361,fff85bda-5628-5fd7-2070-64139d5f5237,completed,"Influenza, seasonal, injectable, preservative ...",140,http://hl7.org/fhir/sid/cvx,3972692e-4dd9-7be8-ccef-92258fc199cc,122ea1c4-d184-e748-edaf-ceccf47144a3,2019-02-08T07:20:37-05:00,True,LINCOLN COUNTY MEDICAL CLINIC
17362,fffaa27d-5bc9-0383-3b48-4047883f10d2,completed,"HPV, quadrivalent",62,http://hl7.org/fhir/sid/cvx,9b71153d-67cb-9411-37b7-1e8bdf804f28,2d312b11-0035-95d2-a2d5-53587e1500e1,2020-08-13T19:46:50-04:00,True,SMITH INTERNAL MEDICINE PA
17363,fffaee9d-c978-5dfc-d27b-78dbd60db2d2,completed,"Influenza, seasonal, injectable, preservative ...",140,http://hl7.org/fhir/sid/cvx,a9964e41-645e-5f0c-6043-039e4e11fe11,fb2b608a-6f83-6b39-737d-388a1f9fdbe7,2014-06-02T19:13:52-04:00,True,KANSAS MEDICAL CLINIC PA
17364,ffff5eb2-af0e-24e0-f5df-138cc0304abc,completed,"Influenza, seasonal, injectable, preservative ...",140,http://hl7.org/fhir/sid/cvx,5664031d-7165-6981-9a8d-2dd7228c1201,bf011023-342c-bdef-5acd-2fc3b503df78,2021-01-12T21:47:29-05:00,True,FAMILY MEDCENTERS PA


### **Step 2**: Figuring out whether the status or the primary source column should filter out immunizations.

In [28]:
connection.execute('''
SELECT
    DISTINCT status,
    primary_source
FROM
    stg_immunization
''').df()

,status,primary_source
0,completed,True


### **Step 3:** Classifying whether any of the vaccines have the word flu in them, and counting the number of unique lifetime recipients of the vaccine.

In [26]:
connection.execute('''
SELECT
    vaccine_name,
    CASE
        WHEN LOWER(vaccine_name) LIKE '%flu%' THEN True ELSE False
        END
    AS flu_vaccine,
    COUNT(DISTINCT patient_id)
FROM
    stg_immunization
GROUP BY
    1,2
''').df()

,vaccine_name,flu_vaccine,count(DISTINCT patient_id)
0,Hib (PRP-OMP),False,169
1,"pneumococcal conjugate vaccine, 13 valent",False,1
2,varicella,False,210
3,"pneumococcal polysaccharide vaccine, 23 valent",False,112
4,MMR,False,210
5,"SARS-COV-2 (COVID-19) vaccine, mRNA, spike pro...",False,292
6,"SARS-COV-2 (COVID-19) vaccine, mRNA, spike pro...",False,363
7,"zoster vaccine, live",False,134
8,"Hep B, adolescent or pediatric",False,156
9,"Influenza, seasonal, injectable, preservative ...",True,1136


### **Step 4:** I'm confused about > 1000 patients that have had the flu vaccine, when my large dataset should only be looking at data for 1,000 patients. Frankly, I don't have a good explanation for this. The one I can think of is that perhaps the ***patient data*** is limited to ~1000 and not the other resource types. But instead, upon further querying, I found that the unique patient count in this table matches the unique patient count in stg_patient.

In [29]:
connection.execute('''
SELECT
    COUNT(*) AS all_rows,
    COUNT(DISTINCT patient_id) AS all_unique_patients
FROM
    stg_immunization
''').df()

,all_rows,all_unique_patients
0,17366,1144


In [31]:
connection.execute('''
SELECT
    COUNT(DISTINCT patient_id) AS all_unique_patients
FROM
    stg_patient
''').df()

,all_unique_patients
0,1144


### **ANSWER 01**
1,136 patients in this dataset have received a flu vaccine.

### **ANSWER 02**
The below query will order the same query as I had above. This treats the "most common vaccine" definition as "immunizations with most # of unique patients." That makes more sense to me than just counting the raw # of vaccines administered.

In [33]:
connection.execute('''
SELECT
    vaccine_name,
    CASE
        WHEN LOWER(vaccine_name) LIKE '%flu%' THEN True ELSE False
        END
    AS flu_vaccine,
    COUNT(DISTINCT patient_id),
    COUNT(DISTINCT immunization_id)
FROM
    stg_immunization
GROUP BY
    1,2
ORDER BY
    3 DESC
LIMIT 5
''').df()

,vaccine_name,flu_vaccine,count(DISTINCT patient_id),count(DISTINCT immunization_id)
0,"Influenza, seasonal, injectable, preservative ...",True,1136,9264
1,"Td (adult), 5 Lf tetanus toxoid, preservative ...",False,741,768
2,"SARS-COV-2 (COVID-19) vaccine, mRNA, spike pro...",False,363,725
3,meningococcal MCV4P,False,293,415
4,"SARS-COV-2 (COVID-19) vaccine, mRNA, spike pro...",False,292,584
